# 🧠 Recursive Insight Agent Notebook
This notebook processes ChatGPT JSON exports and PDFs into recursive insights using a local LLM stub.


In [2]:
# 📦 Imports & Config
import json
from pathlib import Path
from typing import List, Dict
from pprint import pprint
import fitz  # PyMuPDF for PDF support

# 📁 Paths
CHATGPT_JSON_PATH = Path("C:/Users/ANN/Hub/conversations.json")
PDF_PATH = Path("NotebookLLM.pdf")


ModuleNotFoundError: No module named 'fitz'

In [ ]:
# 📥 Input Loader: ChatGPT JSON
def load_chatgpt_json(path: Path) -> List[str]:
    with open(path, 'r', encoding='utf-8') as f:
        all_convos = json.load(f)

    messages = []
    for convo in all_convos:
        mapping = convo.get("mapping", {})
        for node in mapping.values():
            msg_obj = node.get("message", {})
            content = msg_obj.get("content", {}).get("parts", [])
            if isinstance(content, list):
                messages.extend(content)
            elif isinstance(content, str):
                messages.append(content)
    return messages


In [ ]:
# 📥 Input Loader: PDF
def load_pdf_text(path: Path) -> List[str]:
    doc = fitz.open(str(path))
    text_chunks = [page.get_text() for page in doc]
    return text_chunks


In [ ]:
# 🧠 Insight Extractor
def extract_insights(messages: List[str]) -> List[str]:
    insights = []
    for msg in messages:
        if len(msg.strip()) > 50:
            insights.append("🧠 " + msg[:200].strip() + ("..." if len(msg) > 200 else ""))
    return insights


In [ ]:
# 🤖 LLM Stub
def query_llama(prompt: str) -> str:
    return f"[LLM Insight Placeholder]: {prompt[:100]}..."


In [ ]:
# 🧾 Output Formatter
def markdown_output(insights: List[str], responses: List[str]) -> str:
    lines = []
    for i, (insight, response) in enumerate(zip(insights, responses)):
        lines.append(f"### 🧠 Insight {i+1}\n{insight}\n\n**LLM Response:**\n{response}\n")
    return "\n".join(lines)


In [ ]:
# 💾 Optional Save to Markdown
def save_output_markdown(text: str, filename: str = "llm_insights.md"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"✅ Markdown output saved to: {filename}")


In [ ]:
# 🚀 Run (Choose Input Type)
source = "json"  # change to "pdf" if needed

if source == "json":
    messages = load_chatgpt_json(CHATGPT_JSON_PATH)
elif source == "pdf":
    messages = load_pdf_text(PDF_PATH)

insights = extract_insights(messages)
llm_responses = [query_llama(i) for i in insights[:5]]  # limit to 5
md = markdown_output(insights[:5], llm_responses)
print(md)
save_output_markdown(md)
